# 🍎 Health Calculator Agent Tutorial 🍏

Welcome to the **Health Calculator Agent** tutorial, where we'll showcase how to:
1. **Initialize** a project and use the Azure AI Foundry ecosystem
2. **Create an Agent** with **Code Interpreter** capabilities
3. **Perform BMI calculations** and **analyze nutritional data** with sample CSV files
4. **Generate** basic health insights and disclaimers

## Let's Dive In
We'll walk step-by-step, similar to our **Fun & Fit** sample, but with a focus on using **Code Interpreter** for numeric calculations and data analysis. Let's begin!


## 1. Initial Setup
We'll start by importing libraries, loading environment variables, and initializing an **AIProjectClient**. We'll also create a sample CSV for demonstration.


In [ ]:
! pip install pandas

In [ ]:
# Import required libraries
import os  # For environment variables and path operations
import time  # For adding delays if needed
from pathlib import Path  # For cross-platform path handling
import json
import pandas as pd

# Import Azure and utility libraries
from azure.identity import DefaultAzureCredential  # For Azure authentication
from azure.ai.projects import AIProjectClient  # Main client for AI Projects
from azure.ai.agents.models import CodeInterpreterTool


def find_cred_json(start_path):
    current = Path(start_path)
    while current != current.parent:
        cred_file = current / 'cred.json'
        if cred_file.exists():
            return str(cred_file)
        current = current.parent
    return None

start_dir = os.getcwd()  # Use current directory
file_path = find_cred_json(start_dir)

print(f"Found cred.json at: {file_path}")

try:
    with open(file_path, 'r') as f:
        loaded_config = json.load(f)

    print("Project Connection String:", loaded_config['PROJECT_CONNECTION_STRING'])
    print("Tenant ID:", loaded_config['TENANT_ID'])
    print("Model Deployment ID:", loaded_config['MODEL_DEPLOYMENT_NAME'])

    project_client = AIProjectClient(
    credential=DefaultAzureCredential(),
    endpoint=loaded_config["PROJECT_ENDPOINT"],
    )

    print("✅ Successfully initialized AIProjectClient")

except FileNotFoundError:
    print(f"❌ Could not find file at: {file_path}")
except json.JSONDecodeError:
    print("❌ File exists but contains invalid JSON")
except TypeError:
    print("❌ File path was None — cred.json not found in any parent directories.")


# Create sample CSV data for demonstration
def create_sample_data():
    try:
        data = {
            'Date': pd.date_range(start='2024-01-01', periods=7),
            'Calories': [2100, 1950, 2300, 2050, 1900, 2200, 2150],
            'Protein_g': [80, 75, 85, 78, 72, 82, 79],
            'Carbs_g': [250, 230, 270, 245, 225, 260, 255],
            'Fat_g': [70, 65, 75, 68, 63, 73, 71],
            'Fiber_g': [25, 22, 28, 24, 21, 26, 23]
        }
        df = pd.DataFrame(data)
        filename = "nutrition_data.csv"
        df.to_csv(filename, index=False)
        print(f"📄 Created sample data file: {filename}")
        return filename
    except Exception as e:
        print(f"❌ Error creating sample data: {e}")
        return None

sample_file = create_sample_data()

## 2. Create Health Calculator Agent 👩‍💻
We'll upload our sample CSV and then create an agent with **Code Interpreter** enabled. This agent can read the file, run Python code, and return results and visualizations.


In [ ]:
from azure.ai.agents import AgentsClient
from azure.identity import DefaultAzureCredential

agents_client = AgentsClient(
    endpoint=loaded_config["PROJECT_ENDPOINT"],
    credential=DefaultAzureCredential()
)
print("✅ AgentsClient initialized")

In [ ]:
def create_health_calculator(file_path: str):
    """Create an agent with Code Interpreter for health/nutrition calculations."""
    try:
        # ✅ agents_client for file upload
        uploaded_file = agents_client.files.upload_and_poll(
            file_path=file_path,
            purpose="assistants"
        )
        print(f"✅ Uploaded CSV file, ID: {uploaded_file.id}")

        code_tool = CodeInterpreterTool(file_ids=[uploaded_file.id])

        # ✅ agents_client for agent creation
        agent = agents_client.create_agent(
            model=loaded_config["MODEL_DEPLOYMENT_NAME"],
            name="health-calculator-agent",
            instructions=(
                "You are a health calculator agent that can:\n"
                "1. Calculate and interpret BMI\n"
                "2. Analyze nutrition data\n"
                "3. Generate charts/plots\n"
                "4. Always include medical disclaimers"
            ),
            tools=code_tool.definitions,
            tool_resources=code_tool.resources
        )
        print(f"🎉 Agent created, ID: {agent.id}")
        return agent, uploaded_file
    except Exception as e:
        print(f"❌ Error creating health calculator agent: {e}")
        return None, None

health_agent, uploaded_file = None, None
if 'sample_file' in globals() and sample_file:
    health_agent, uploaded_file = create_health_calculator(sample_file)

## 3. BMI Calculation with Code Interpreter
We'll create a thread for BMI calculations. We'll feed in the user's height/weight, and ask the agent to show how it calculates BMI, interpret the result, and always disclaim professional advice.


In [ ]:
import time

def calculate_bmi_with_agent(agent, height_inches, weight_pounds):
    """Calculate BMI using the code interpreter agent."""
    try:
        thread = agents_client.threads.create()  # ✅
        print(f"📝 Created thread for BMI calculation, ID: {thread.id}")

        user_text = (
            f"Calculate BMI for \n"
            f"Height: {height_inches} inches\n"
            f"Weight: {weight_pounds} pounds\n"
            "Please: \n"
            "1. Show calculation \n"
            "2. Interpret the result \n"
            "3. Include disclaimers \n"
        )

        msg = agents_client.messages.create(  # ✅
            thread_id=thread.id,
            role="user",
            content=user_text
        )
        print(f"➕ Created BMI request message, ID: {msg.id}")

        run = agents_client.runs.create(  # ✅
            thread_id=thread.id,
            agent_id=agent.id
        )
        print(f"▶️ Started run, ID: {run.id}, initial status: {run.status}")

        while run.status in ["queued", "in_progress", "requires_action"]:
            time.sleep(1)
            run = agents_client.runs.get(  # ✅
                thread_id=thread.id,
                run_id=run.id
            )

        print(f"🤖 BMI run finished with status: {run.status}")
        
        if run.status == "completed":
            messages = agents_client.messages.list(thread_id=thread.id)  # ✅
            message_list = list(messages)
            
            for message in message_list:
                if message.role == "assistant":
                    for content in message.content:
                        if hasattr(content, 'text') and content.text:
                            print(f"\n🤖 BMI Analysis:\n{content.text.value}")
        
        return thread, run

    except Exception as e:
        print(f"❌ Error during BMI calculation: {e}")
        return None, None

if health_agent:
    bmi_thread, bmi_run = calculate_bmi_with_agent(health_agent, 70, 180)

## 4. Nutrition Analysis
We'll create another thread where the user can ask the agent to analyze the **`nutrition_data.csv`** we uploaded. The agent can read the file, compute macros, produce charts, and disclaim that it's not offering personalized medical advice.


In [ ]:
import time

def analyze_nutrition_data(agent):
    """Ask the agent to analyze the uploaded nutrition data."""
    try:
        thread = agents_client.threads.create()  # ✅
        print(f"📝 Created thread for nutrition analysis, ID: {thread.id}")

        user_text = (
            "Analyze the CSV file with daily nutrition data.\n"
            "1. Compute average daily macros (calories, protein, carbs, fat, fiber).\n"
            "2. Create a chart to show trends.\n"
            "3. Discuss any insights or disclaimers.\n"
        )

        msg = agents_client.messages.create(  # ✅
            thread_id=thread.id,
            role="user",
            content=user_text
        )
        print(f"➕ Created nutrition request message, ID: {msg.id}")

        run = agents_client.runs.create(  # ✅
            thread_id=thread.id,
            agent_id=agent.id
        )
        print(f"▶️ Started run, ID: {run.id}, initial status: {run.status}")

        while run.status in ["queued", "in_progress", "requires_action"]:
            time.sleep(1)
            run = agents_client.runs.get(  # ✅
                thread_id=thread.id,
                run_id=run.id
            )

        print(f"🤖 Nutrition run finished with status: {run.status}")
        
        if run.status == "completed":
            messages = agents_client.messages.list(thread_id=thread.id)  # ✅
            message_list = list(messages)
            
            for message in message_list:
                if message.role == "assistant":
                    for content in message.content:
                        if hasattr(content, 'text') and content.text:
                            print(f"\n🤖 Nutrition Analysis:\n{content.text.value}")
                        elif hasattr(content, 'image_file'):
                            print(f"\n📊 Chart generated: {content.image_file.file_id}")
        elif run.status == "failed":
            print("❌ Run failed. Checking for error details...")
            if hasattr(run, 'last_error') and run.last_error:
                print(f"   Error Code: {run.last_error.code}")
                print(f"   Error Message: {run.last_error.message}")
        
        return thread, run

    except Exception as e:
        print(f"❌ Error analyzing nutrition data: {e}")
        import traceback
        traceback.print_exc()
        return None, None

if health_agent:
    nutrition_thread, nutrition_run = analyze_nutrition_data(health_agent)

## 5. Viewing Results & Visualizations 📊
The agent may produce text insights, disclaimers, and even images with charts. Let's fetch them from our threads!


In [ ]:
def view_agent_responses(thread_id, save_to_jupyter_root=True):
    try:
        messages = agents_client.messages.list(thread_id=thread_id)  # ✅
        message_list = list(messages)
        
        print("\n🔎 Agent Responses:")
        for msg in message_list:
            if msg.role == "assistant" and msg.content:
                for c in msg.content:
                    if hasattr(c, "text"):
                        print("Response:", c.text.value, "\n")
                    elif hasattr(c, "image_file"):
                        file_id = c.image_file.file_id
                        
                        if save_to_jupyter_root:
                            import os
                            jupyter_root = os.getcwd()
                            outname = os.path.join(jupyter_root, f"chart_{file_id}.png")
                        else:
                            outname = f"chart_{file_id}.png"
                        
                        file_content_stream = agents_client.files.get_content(file_id)  # ✅
                        with open(outname, "wb") as file:
                            for chunk in file_content_stream:
                                file.write(chunk)
                        print(f"🖼️ Saved image output: {outname}")

    except Exception as e:
        print(f"❌ Error viewing agent responses: {e}")
        import traceback
        traceback.print_exc()

if bmi_thread and bmi_run:
    print("\n=== BMI Calculation Results ===")
    view_agent_responses(bmi_thread.id, save_to_jupyter_root=True)

if nutrition_thread and nutrition_run:
    print("\n=== Nutrition Analysis Results ===")
    view_agent_responses(nutrition_thread.id, save_to_jupyter_root=True)

## 6. Cleanup & Best Practices
We can remove our agent and sample data if desired. In production, you might keep them for repeated usage.

### Best Practices in a Nutshell
1. **Data Handling** – Validate input data, handle missing values, properly manage file attachments.
2. **Calculations** – Provide formula steps, disclaimers, limit scope to general wellness, remind user you're not a doctor.
3. **Visualizations** – Use clear labeling and disclaimers that charts are for educational demonstrations.
4. **Security** – Monitor usage, limit access to code interpreter if dealing with proprietary data.


In [ ]:
import os

def cleanup_all_correct():
    """Cleanup function with correct method names."""
    
    try:
        if 'uploaded_file' in globals() and uploaded_file:
            agents_client.files.delete(uploaded_file.id)  # ✅
            print("🗑️ Deleted uploaded file from agent service.")
    except Exception as e:
        print(f"ℹ️ Uploaded file cleanup: {e}")

    try:
        if 'health_agent' in globals() and health_agent:
            agents_client.delete_agent(health_agent.id)  # ✅
            print("🗑️ Deleted health calculator agent.")
    except Exception as e:
        print(f"ℹ️ Agent cleanup: {e}")

    try:
        if 'sample_file' in globals() and sample_file and os.path.exists(sample_file):
            os.remove(sample_file)
            print("🗑️ Deleted local sample CSV file.")
    except Exception as e:
        print(f"⚠️ Could not delete local file: {e}")

    print("✅ Cleanup completed!")


def list_current_resources():
    """List current agents and files."""
    try:
        print("📋 Current agents:")
        agents = agents_client.list_agents()  # ✅
        agent_list = list(agents)
        if agent_list:
            for agent in agent_list:
                print(f"   - {agent.name} (ID: {agent.id})")
        else:
            print("   No agents found")
    except Exception as e:
        print(f"   Error listing agents: {e}")

    try:
        print("\n📋 Current files:")
        files = agents_client.files.list()  # ✅
        file_list = list(files)
        if file_list:
            for file in file_list:
                print(f"   - {file.filename} (ID: {file.id})")
        else:
            print("   No files found")
    except Exception as e:
        print(f"   Error listing files: {e}")


print("=== Current Resources ===")
list_current_resources()

print("\n=== Running Cleanup ===")
cleanup_all_correct()

print("\n=== Resources After Cleanup ===")
list_current_resources()

# Congratulations! 🎉
You now have a **Health Calculator Agent** with the **Code Interpreter** tool that can:
- Perform **BMI calculations** and disclaim that it's not a doctor.
- **Analyze** simple CSV-based nutrition data and produce insights + charts.
- Return images (charts) and text-based insights.

Happy (healthy) coding! 💪
